In this file, we
1. preprocess a file and
2. create a multiplex horizontal visibility graph out of a 30s time window.

The time window starts in an interictal state and ends in an ictal state, allowing for analysis of the community evolution over time.

Comments:
- File used: chb01_03.edf
- Time window: 2980s (interictal) - 3010s (ictal)
- The output can be found in data/graphs under "inter_to_ict_chb01_03_2980_3010"
- The threshold for intra layer edges is raised to 0.1 (instead of 0.05) in order to have stronger results. Still, the inter-layer edges (n=1'943'040) outweigh the intra-layer edges (n=35'161) by far. This should be taken into account for the analysis. I didn't raise it further in order to avoid even bigger files, but if needed, a new file can be created by adjusting the code (in the last cell "intra_keep_ratio").

In [25]:
#!pip install ts2vg mne

from pathlib import Path
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import save_npz
from ts2vg import HorizontalVG
import mne

In [28]:
# preprocessing function
def get_seizure_period(annotation_file):
    with open(annotation_file, "rb") as f:
        byte_array = f.read()
    start = int(bin(byte_array[38])[2:] + bin(byte_array[41])[2:], 2)
    length = byte_array[49]
    return start, length

def process_edf_with_labels(edf_file, seizure_file=None):
    edf_file = Path(edf_file)
    raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)
    raw.filter(1., 45.)
    raw.set_eeg_reference('average')

    sfreq = raw.info["sfreq"]
    n_samples = len(raw.times)
    time_marks = np.arange(n_samples) / sfreq
    electrode_data = raw.get_data()
    electrode_names = raw.ch_names

    df = pd.DataFrame(electrode_data.T, columns=electrode_names)
    df.index = time_marks
    df.index.name = "Time (s)"

    if seizure_file is not None:
        seizure_start, seizure_length = get_seizure_period(Path(seizure_file))
        seizure_end = seizure_start + seizure_length
        df["label"] = df.index.map(
            lambda t: "ictal" if seizure_start <= t <= seizure_end else "interictal"
        )
    else:
        df["label"] = "interictal"

    return df

In [29]:
# navigate to your individual path
%cd "C:\Users\marin\OneDrive\Dokumente\Uni\MSc_DNs_Fribourg\FS26\Social_Media_Analytics\git"
RAW_DIR = Path("data/raw")

# preprocess file
df = process_edf_with_labels(
    edf_file=RAW_DIR / "chb01_03.edf",
    seizure_file=RAW_DIR / "chb01_03.edf.seizures"
)

print(df.shape)
print(df["label"].value_counts())

C:\Users\marin\OneDrive\Dokumente\Uni\MSc_DNs_Fribourg\FS26\Social_Media_Analytics\git


C:\Users\marin\AppData\Local\Temp\ipykernel_38636\3197021929.py:11: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 845 samples (3.301 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
(921600, 24)
label
interictal    911359
ictal          10241
Name: count, dtype: int64


In [30]:
# extract window
def extract_window_by_time(df, start_time_s, window_size_s=30):
    dt = float(df.index[1] - df.index[0])
    sfreq = int(round(1 / dt))
    window_size_samples = int(round(window_size_s * sfreq))

    index_values = np.asarray(df.index, dtype=float)
    start_idx = int(np.argmin(np.abs(index_values - start_time_s)))
    end_idx = start_idx + window_size_samples

    if end_idx > len(df):
        raise ValueError("Window exceeds file length.")

    return df.iloc[start_idx:end_idx].copy(), start_idx

window_df, start_idx = extract_window_by_time(df, start_time_s=2980.0, window_size_s=30)

print(f"Window: {window_df.index[0]:.1f}s → {window_df.index[-1]:.1f}s")
print(f"Samples: {len(window_df)}")
print(f"Label counts:\n{window_df['label'].value_counts()}")

Window: 2980.0s → 3010.0s
Samples: 7680
Label counts:
label
interictal    4096
ictal         3584
Name: count, dtype: int64


In [47]:
# prune edges
def prune_edges_by_quantile(edge_rows, keep_ratio=0.1):
    if not edge_rows:
        return []
    if keep_ratio is None:
        return edge_rows
    if not (0 < keep_ratio <= 1):
        raise ValueError("keep_ratio must be in (0, 1].")

    edge_df = pd.DataFrame(edge_rows)
    threshold = edge_df["weight"].quantile(1 - keep_ratio)
    return edge_df[edge_df["weight"] >= threshold].to_dict("records")

In [48]:
# functions
def build_multiplex_hvg(window_df, intra_keep_ratio=0.1):
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_electrodes = len(electrode_cols)
    n_timepoints = len(window_df)
    n_nodes_total = n_electrodes * n_timepoints

    print(f"Electrodes        : {n_electrodes}")
    print(f"Time points       : {n_timepoints}")
    print(f"Total nodes       : {n_nodes_total}")

    def node_id(electrode, t):
        return f"{electrode}_t{t}"

    node_labels = [
        node_id(electrode, t)
        for electrode in electrode_cols
        for t in range(n_timepoints)
    ]
    label_to_idx = {label: idx for idx, label in enumerate(node_labels)}
    node_index = {
        (electrode, t): node_id(electrode, t)
        for electrode in electrode_cols
        for t in range(n_timepoints)
    }

    adj_sparse = sp.lil_matrix((n_nodes_total, n_nodes_total), dtype=np.float32)

    intra_rows = []
    layer_info = {}

    # 1) Intra-layer HVG edges (weighted + pruned)
    for layer_idx, electrode in enumerate(electrode_cols):
        ts = window_df[electrode].to_numpy()
        hvg = HorizontalVG()
        hvg.build(ts)

        layer_intra_rows = []
        for t_i, t_j in hvg.edges:
            t_i, t_j = int(t_i), int(t_j)
            u = node_id(electrode, t_i)
            v = node_id(electrode, t_j)
            weight = abs(float(ts[t_i]) - float(ts[t_j]))
            layer_intra_rows.append({
                "Source": u, "Target": v,
                "source_label": u, "target_label": v,
                "edge_type": "intra", "layer": electrode,
                "weight": weight
            })

        n_before = len(layer_intra_rows)

        # apply pruning if requested
        if intra_keep_ratio is not None:
            layer_intra_rows = prune_edges_by_quantile(layer_intra_rows, keep_ratio=intra_keep_ratio)

        n_after = len(layer_intra_rows)

        for row in layer_intra_rows:
            intra_rows.append(row)
            i = label_to_idx[row["Source"]]
            j = label_to_idx[row["Target"]]
            adj_sparse[i, j] = row["weight"]
            adj_sparse[j, i] = row["weight"]

        layer_info[electrode] = {
            "layer_index": int(layer_idx),
            "n_intra_edges_before": int(n_before),
            "n_intra_edges_after": int(n_after)
        }

    print(f"Intra-layer edges : {len(intra_rows)}")

    # 2) Inter-layer edges
    inter_rows = []
    for t in range(n_timepoints):
        for i in range(n_electrodes):
            for j in range(i + 1, n_electrodes):
                e_i = electrode_cols[i]
                e_j = electrode_cols[j]
                u = node_id(e_i, t)
                v = node_id(e_j, t)
                inter_rows.append({
                    "Source": u, "Target": v,
                    "source_label": u, "target_label": v,
                    "edge_type": "inter", "layer": f"{e_i}<->{e_j}",
                    "weight": 1.0
                })
                adj_sparse[label_to_idx[u], label_to_idx[v]] = 1.0
                adj_sparse[label_to_idx[v], label_to_idx[u]] = 1.0

    print(f"Inter-layer edges : {len(inter_rows)}")

    edge_list = pd.DataFrame(intra_rows + inter_rows)
    print(f"Total edges       : {len(edge_list)}")

    return edge_list, adj_sparse.tocsr(), node_index, layer_info, node_labels


def save_multiplex_hvg_outputs(
    edge_list, adj_sparse, node_index, node_labels,
    layer_info, window_df, graph_id, source_file,
    window_id, label, intra_keep_ratio=0.05, out_root="data/graphs"
):
    out_root = Path(out_root)
    edges_dir = out_root / "edges"
    nodes_dir = out_root / "nodes"
    adj_dir   = out_root / "adjacency_sparse"
    meta_dir  = out_root / "metadata"
    windows_dir = out_root / "windows"

    for d in [edges_dir, nodes_dir, adj_dir, meta_dir, windows_dir]:
        d.mkdir(parents=True, exist_ok=True)

    node_df = make_multiplex_node_table(window_df, graph_label=label)

    edge_path = edges_dir / f"{graph_id}_edges.csv"
    edge_list.to_csv(edge_path, index=False)

    node_path = nodes_dir / f"{graph_id}_nodes.csv"
    node_df.to_csv(node_path, index=False)

    adj_path = adj_dir / f"{graph_id}_adjacency_sparse.npz"
    save_npz(adj_path, adj_sparse)

    window_path = windows_dir / f"{graph_id}_window.parquet"
    window_df.to_parquet(window_path)

    node_labels_path = meta_dir / f"{graph_id}_node_labels.json"
    with open(node_labels_path, "w", encoding="utf-8") as f:
        json.dump(node_labels, f, indent=2)

    layer_info_df = (
        pd.DataFrame(layer_info).T.reset_index().rename(columns={"index": "electrode"})
    )
    layer_info_path = meta_dir / f"{graph_id}_layer_info.csv"
    layer_info_df.to_csv(layer_info_path, index=False)

    node_index_str = {f"{k[0]}__t{k[1]}": v for k, v in node_index.items()}
    node_index_path = meta_dir / f"{graph_id}_node_index.json"
    with open(node_index_path, "w", encoding="utf-8") as f:
        json.dump(node_index_str, f, indent=2)

    metadata = {
        "graph_id": graph_id,
        "source_file": str(source_file),
        "window_id": window_id,
        "label": label,
        "n_nodes": int(node_df.shape[0]),
        "n_edges": int(edge_list.shape[0]),
        "n_intra_edges": int((edge_list["edge_type"] == "intra").sum()),
        "n_inter_edges": int((edge_list["edge_type"] == "inter").sum()),
        "n_layers": int(node_df["electrode"].nunique()),
        "n_timepoints": int(len(window_df)),
        "window_start_time_s": float(window_df.index[0]),
        "window_end_time_s": float(window_df.index[-1]),
        "adjacency_format": "scipy_sparse_npz",
        "intra_edge_weight": "abs(amplitude_i - amplitude_j)",
        "intra_keep_ratio": intra_keep_ratio
    }

    metadata_path = meta_dir / f"{graph_id}_metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return {
        "edges": str(edge_path),
        "nodes": str(node_path),
        "adjacency_sparse": str(adj_path),
        "window": str(window_path),
        "node_labels": str(node_labels_path),
        "layer_info": str(layer_info_path),
        "node_index": str(node_index_path),
        "metadata": str(metadata_path),
    }

In [51]:
# apply the functions

GRAPH_DIR = Path("data/graphs")
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

graph_id  = "inter_to_ict_chb01_03_2980_3010"
window_id = "inter_to_ict_chb01_03_2980_3010"

edge_list, adj_sparse, node_index, layer_info, node_labels = build_multiplex_hvg(
    window_df,
    intra_keep_ratio=0.1   # top 5% strongest edges kept; set None to keep all
)

saved_paths = save_multiplex_hvg_outputs(
    edge_list=edge_list,
    adj_sparse=adj_sparse,
    node_index=node_index,
    node_labels=node_labels,
    layer_info=layer_info,
    window_df=window_df,
    graph_id=graph_id,
    source_file=RAW_DIR / "chb01_03.edf",
    window_id=window_id,
    label="ictal",
    intra_keep_ratio=0.1,
    out_root=GRAPH_DIR
)

print(saved_paths)

Electrodes        : 23
Time points       : 7680
Total nodes       : 176640
Intra-layer edges : 35161
Inter-layer edges : 1943040
Total edges       : 1978201
{'edges': 'data\\graphs\\edges\\inter_to_ict_chb01_03_2980_3010_edges.csv', 'nodes': 'data\\graphs\\nodes\\inter_to_ict_chb01_03_2980_3010_nodes.csv', 'adjacency_sparse': 'data\\graphs\\adjacency_sparse\\inter_to_ict_chb01_03_2980_3010_adjacency_sparse.npz', 'window': 'data\\graphs\\windows\\inter_to_ict_chb01_03_2980_3010_window.parquet', 'node_labels': 'data\\graphs\\metadata\\inter_to_ict_chb01_03_2980_3010_node_labels.json', 'layer_info': 'data\\graphs\\metadata\\inter_to_ict_chb01_03_2980_3010_layer_info.csv', 'node_index': 'data\\graphs\\metadata\\inter_to_ict_chb01_03_2980_3010_node_index.json', 'metadata': 'data\\graphs\\metadata\\inter_to_ict_chb01_03_2980_3010_metadata.json'}
